# 04. 104개에서 80개 Expert 모델 만들기

03번에서 만든 정렬된 104개 Expert 모델의 각 층에서 **정렬 ID 0~79번**을 남기고 80~103번을 제거합니다. 정렬 ID와 원래 모델의 Expert ID는 다르며, 내보낸 manifest에 각 층의 대응 관계가 기록됩니다.

이 노트북은 저장소 루트 또는 `ipynb` 폴더에서 실행합니다. 기존 104개 모델과 해당 manifest, 정렬 manifest가 필요합니다. 아래 설정의 `RUN_EXPORT`를 `True`로 바꾼 뒤 셀을 순서대로 실행하세요.

저장된 기존 셀 출력은 이전 환경 기록이며 현재 A100 실행 결과가 아닙니다.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

ROOT = Path.cwd().resolve()
if ROOT.name == 'ipynb':
    ROOT = ROOT.parent
assert (ROOT / 'moe_prune').is_dir(), '저장소 루트 또는 ipynb 폴더에서 실행하세요.'
sys.path.insert(0, str(ROOT))
from tools import prune_e80 as work
from tools import manual80

INSTALL_DEPENDENCIES = False
PREPARE_GRADER = False
PREPARE_ENGINE = False
BUILD_ENGINE = False
RUN_EXPORT = False
RUN_GENERATION = False
RUN_GRADING = False
RUN_PLOTS = False
SOURCE = work.SOURCE
OUTPUT = work.OUTPUT
RUN = work.RUN
GENERATION_DIR = manual80.DIRECTORY
SOURCE_MANIFEST = work.PARENT
REORDER_MANIFEST = work.REORDER
# 2026-11-09: 경로를 변경해도 검증된 104개 원본 모델의 식별 기준은 유지한다.
EXPECTED_SHA = work.SOURCE_SHA
EXPECTED_BYTES = work.SOURCE_BYTES
EXPECTED_LAYERS = list(range(30))

print({'104 모델': str(SOURCE), '있음': SOURCE.is_file()})
print({'104 manifest': str(SOURCE_MANIFEST), '있음': SOURCE_MANIFEST.is_file()})
print({'정렬 manifest': str(REORDER_MANIFEST), '있음': REORDER_MANIFEST.is_file()})
print({'80 모델 출력': str(OUTPUT), '이미 있음': OUTPUT.exists()})
print({'결과 폴더': str(RUN)})

## Colab 환경 준비

설치·CUDA 빌드·공식 채점기 준비는 플래그를 켤 때만 실행합니다.

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', '--isolated', 'install',
                    '-r', str(ROOT / 'requirements-research.txt')], check=True)
if PREPARE_GRADER:
    from moe_prune.benchmark_data import prepare_grader
    prepare_grader(ROOT)
if PREPARE_ENGINE:
    command = [sys.executable, str(ROOT / 'tools/native/prepare.py'),
               '--backend', 'cuda', '--gpu-target', '80']
    if BUILD_ENGINE:
        command.append('--execute')
    subprocess.run(command, cwd=ROOT, check=True)
if not any((INSTALL_DEPENDENCIES, PREPARE_GRADER, PREPARE_ENGINE)):
    print('환경 준비를 실행하지 않았습니다.')


## 80개 모델 생성

모델과 manifest를 검증하고 각 층에서 0~79번을 남겨 새 GGUF를 만든 뒤, 모든 tensor의 바이트를 확인합니다. 전체 파일 해시를 계산하므로 시간이 걸리고 출력 모델을 저장할 디스크 공간이 필요합니다.

기존 출력 모델, 부분 파일 또는 결과 manifest가 있으면 덮어쓰지 않고 중단합니다. 다시 생성하려면 위 `OUTPUT`과 `RUN`을 사용하지 않은 새 경로로 바꾸세요. 파일 검증 통과는 추론 동등성이나 모델 품질 검증을 뜻하지 않습니다.

In [ ]:
if RUN_EXPORT:
    result = work._export_e80(
        source=SOURCE, output=OUTPUT, run=RUN,
        source_manifest=SOURCE_MANIFEST, reorder_manifest=REORDER_MANIFEST,
        expected_sha=EXPECTED_SHA, expected_bytes=EXPECTED_BYTES,
        expected_layers=EXPECTED_LAYERS,
    )
    print({key: result[key] for key in (
        'output', 'expert_count', 'top_k', 'output_bytes', 'output_sha256',
        'file_validation_status', 'inference_status', 'quality_status', 'runtime_equivalence',
    )})
else:
    print('모델 생성 대기 중: RUN_EXPORT=True로 설정하세요.')

## 기존 결과 확인

결과 manifest가 있으면 저장된 검증 상태와 모델 정보를 표시합니다. 이 표시는 모델 파일의 현재 해시를 다시 계산하지 않습니다.

In [ ]:
manifest_path = RUN / 'export-manifest.json'
if manifest_path.is_file():
    info = json.loads(manifest_path.read_text(encoding='utf-8'))
    print({key: info.get(key) for key in (
        'output', 'expert_count', 'top_k', 'output_bytes', 'output_sha256',
        'file_validation_status', 'inference_status', 'quality_status', 'runtime_equivalence',
    )})
else:
    print('저장된 결과 manifest가 없습니다.')

## 생성과 채점

104개 모델에서 80개 모델을 먼저 만든 뒤 별도 생성·채점 플래그를 켭니다.
추론은 전체 GPU 적재를 요청하며 RAM 80%·VRAM 97% 초과는 경고합니다.
채점 코드는 추론 종료 후 Colab 사용자 권한의 별도 프로세스에서 실행되며 보안 격리되지 않습니다.
기존 결과 폴더는 덮어쓰지 않습니다.

In [ ]:
if RUN_GENERATION:
    print(manual80.generate(directory=GENERATION_DIR))
if RUN_GRADING:
    print(manual80.grade(directory=GENERATION_DIR))
if RUN_PLOTS:
    print(manual80.plots(directory=GENERATION_DIR))
if not any((RUN_GENERATION, RUN_GRADING, RUN_PLOTS)):
    print('생성·채점·시각화를 실행하지 않았습니다.')